## Step 7 — Chronological Train/Validation/Test Split

**Purpose:** Split the cleaned dataset into train (70%), validation (15%),
and test (15%) sets, strictly in time order, with no shuffling.

**Why:** This is time-series data where each row's target and lag features
depend on surrounding timestamps. Random shuffling would let the model see
"future" data during training, causing data leakage and unrealistically
high accuracy that won't hold in real-world deployment. A chronological
split ensures the model only ever learns from the past to predict the
future, exactly how it would be used in practice.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

folder = "/content/drive/MyDrive/Project2_SolarPV/"

In [ ]:
# STEP 7 — CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT

# Loading the cleaned dataset from Step 6
df_clean = pd.read_csv(folder + "df_clean_step6.csv")
df_clean["DATE_TIME"] = pd.to_datetime(df_clean["DATE_TIME"])

# Making sure it's sorted chronologically
df_clean = df_clean.sort_values("DATE_TIME").reset_index(drop=True)

n = len(df_clean)
print("Total rows:", n)

# Computing split indices: 70% train, 15% val, 15% test
train_end = int(n * 0.70)
val_end   = int(n * 0.85)   # 70% + 15% = 85%

# Slicing sequentially — NO shuffling
train_df = df_clean.iloc[:train_end]
val_df   = df_clean.iloc[train_end:val_end]
test_df  = df_clean.iloc[val_end:]

print(f"Train: {len(train_df)} rows  ({train_df['DATE_TIME'].min()} to {train_df['DATE_TIME'].max()})")
print(f"Val:   {len(val_df)} rows  ({val_df['DATE_TIME'].min()} to {val_df['DATE_TIME'].max()})")
print(f"Test:  {len(test_df)} rows  ({test_df['DATE_TIME'].min()} to {test_df['DATE_TIME'].max()})")

Total rows: 3103
Train: 2172 rows  (2020-05-15 00:45:00 to 2020-06-08 05:00:00)
Val:   465 rows  (2020-06-08 05:15:00 to 2020-06-13 01:15:00)
Test:  466 rows  (2020-06-13 01:30:00 to 2020-06-17 23:30:00)


### Leakage Verification — Confirm No Temporal Overlap

**Purpose:** Programmatically verify that the train, validation, and test
sets do not overlap in time i.e., every timestamp in train comes before
every timestamp in validation, and every timestamp in validation comes
before every timestamp in test.

**Why:** Manually computing split indices (70%/15%/15%) is usually correct,
but silent bugs are possible (e.g., off-by-one errors, unsorted data before
splitting). Rather than assuming the split is leakage-free, this check
proves it with `assert` statements, if any set overlaps in time with
another, the code will stop immediately with a clear error message instead
of silently producing invalid results. This directly enforces Rule 1 of
the project's Data Leakage Prevention Strategy: "Chronological splitting
only, no random shuffle at any stage."

In [ ]:
# Verify NO leakage: splits must not overlap in time
assert train_df["DATE_TIME"].max() < val_df["DATE_TIME"].min(), "LEAKAGE: train/val overlap!"
assert val_df["DATE_TIME"].max() < test_df["DATE_TIME"].min(), "LEAKAGE: val/test overlap!"

print("No temporal overlap between train, validation, and test sets.")

No temporal overlap between train, validation, and test sets.


In [ ]:
train_df.to_csv(folder + "train_step7.csv", index=False)
val_df.to_csv(folder + "val_step7.csv", index=False)
test_df.to_csv(folder + "test_step7.csv", index=False)

print("Train, Val, Test sets saved.")

Train, Val, Test sets saved.
